# Explicabilidad de F1 para OCR con ViT
Este notebook implementa las etapas **A‑1 → A‑6** descritas en la conversación:
1. Construir $X$ (PCs + rasgos visuales) y $y$ (F1)
2. Preprocesado (escalado + one‑hot)
3. Modelo supervisado (XGBoost)
4. Importancia global (Permutation & SHAP)
5. Bootstrap IC 95 %
6. Agrupar la importancia por *rasgo origen* para aislar únicamente las **características visuales**

**Ajusta** las rutas y las listas `manual_cont` / `manual_cat` a tu dataset.

In [ ]:
import numpy as np, pandas as pd, shap, joblib, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBRegressor
import warnings, random, os
warnings.filterwarnings('ignore')

# ---------- CONFIGURACIÓN ----------
DATA_PATH   = '/explainability/data/data.csv'  # <-- edita
EMB_PREFIX  = 'emb_'
PCA_VAR     = 0.95
MAX_PC      = 15
CV_SPLITS   = 5
SEED        = 42
OUTDIR      = './explainability'
os.makedirs(OUTDIR, exist_ok=True)
np.random.seed(SEED); random.seed(SEED)

/home/msi/miniconda3/envs/doctor_saas/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ---------- 1. Carga y PCA ----------
df = pd.read_parquet(DATA_PATH) if DATA_PATH.endswith('.parquet') else pd.read_csv(DATA_PATH)
assert 'f1_img' in df.columns, 'La columna f1_img debe existir'

emb_cols = [c for c in df.columns if c.startswith(EMB_PREFIX)]
pca_full = PCA(n_components=min(MAX_PC, len(emb_cols)), random_state=SEED)
pca_full.fit(df[emb_cols])
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
k_opt   = int(np.searchsorted(cum_var, PCA_VAR)) + 1
print(f'k_opt={k_opt}, var_acum={cum_var[k_opt-1]:.3f}')

PC = pca_full.transform(df[emb_cols])[:, :k_opt]
pc_cols = [f'PC{i+1}' for i in range(k_opt)]
df[pc_cols] = PC
joblib.dump(pca_full, f'{OUTDIR}/pca.joblib')

In [ ]:
# ---------- 2. Feature matrix ----------
manual_cont = ['font_size', 'blur', 'contrast']          # <-- edita
manual_cat  = ['idioma', 'layout']                       # <-- edita

X_cols = pc_cols + manual_cont + manual_cat
y      = df['f1_img'].values

pre = ColumnTransformer([
        ('num', StandardScaler(), pc_cols + manual_cont),
        ('cat', OneHotEncoder(handle_unknown='ignore'), manual_cat)
      ])

model = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8,
                     objective='reg:squarederror', random_state=SEED)
pipe  = make_pipeline(pre, model)

In [ ]:
# ---------- 3. Validación cruzada ----------
cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)
r2  = cross_val_score(pipe, df[X_cols], y, cv=cv, scoring='r2')
mae = -cross_val_score(pipe, df[X_cols], y, cv=cv, scoring='neg_mean_absolute_error')
print(f'R² = {r2.mean():.3f} ± {r2.std():.3f}')
print(f'MAE= {mae.mean():.3f} ± {mae.std():.3f}')

pipe.fit(df[X_cols], y)
joblib.dump(pipe, f'{OUTDIR}/model.joblib')

In [ ]:
# ---------- 4. Importancia por permutación (agrupada) ----------
perm = permutation_importance(pipe, df[X_cols], y, n_repeats=30,
                              random_state=SEED, n_jobs=-1)
feat_names = pipe[:-1].get_feature_names_out()
group_map = {}
# PCs
for i, pc in enumerate(pc_cols):
    group_map.setdefault(pc, []).append(i)
# Continuos manuales
for col in manual_cont:
    j = list(feat_names).index(f'num__{col}')
    group_map.setdefault(col, []).append(j)
# Categóricos manuales
for col in manual_cat:
    for j, name in enumerate(feat_names):
        if name.startswith(f'cat__{col}_'):
            group_map.setdefault(col, []).append(j)

perm_group = [(g, perm.importances_mean[idx].sum(),
               np.sqrt((perm.importances_std[idx]**2).sum()))
              for g, idx in group_map.items()]
perm_df = (pd.DataFrame(perm_group, columns=['feature','mean','std'])
            .sort_values('mean', ascending=False))
perm_df.to_csv(f'{OUTDIR}/perm_importance_GROUPED.csv', index=False)
perm_df.head()

In [ ]:
# ---------- 5. SHAP summary & agrupado ----------
sample = df.sample(n=min(5000, len(df)), random_state=SEED)
X_trans = pipe[:-1].transform(sample[X_cols])
expl    = shap.TreeExplainer(pipe[-1])
sh_vals = expl.shap_values(X_trans)

shap.summary_plot(sh_vals, X_trans, feature_names=feat_names, show=False)
plt.savefig(f'{OUTDIR}/shap_summary.png', bbox_inches='tight')
plt.close()

group_shap = {g: np.abs(sh_vals[:, idx]).mean() for g, idx in group_map.items()}
shap_df = (pd.Series(group_shap, name='mean_abs_shap')
             .sort_values(ascending=False))
shap_df.to_csv(f'{OUTDIR}/shap_GROUPED.csv')
shap_df.head()

In [ ]:
# ---------- 6. Bootstrap IC 95 % ----------
BOOT_ITERS = 1000
coefs_boot = np.zeros((BOOT_ITERS, len(group_map)))
feat_list  = list(group_map.keys())
for i in range(BOOT_ITERS):
    idx = np.random.choice(len(df), len(df), replace=True)
    pipe.fit(df.iloc[idx][X_cols], y[idx])
    expl_i = shap.TreeExplainer(pipe[-1])
    sh_i   = expl_i.shap_values(pipe[:-1].transform(df.iloc[idx][X_cols]))
    for j, g in enumerate(feat_list):
        coefs_boot[i, j] = np.abs(sh_i[:, group_map[g]]).mean()

ci_low  = np.percentile(coefs_boot, 2.5, axis=0)
ci_high = np.percentile(coefs_boot, 97.5, axis=0)
mean_abs = shap_df[feat_list].values

ci_df = pd.DataFrame({'feature': feat_list,
                      'mean': mean_abs,
                      'ci_low': ci_low,
                      'ci_high': ci_high})
ci_df = ci_df.sort_values('mean', ascending=False)
ci_df.to_csv(f'{OUTDIR}/shap_bootstrap_ci.csv', index=False)
ci_df.head()